<a href="https://colab.research.google.com/github/thalesfreirefarias/Controle-Financeiro-Pro-/blob/main/03_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# CONTROLE FINANCEIRO PRO 2.0
# DIA 3 - DASHBOARD FINANCEIRO
# ==========================================

# 1. Importação das bibliotecas

import pandas as pd
import requests
import base64

from io import BytesIO
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.chart import BarChart, DoughnutChart, Reference


# 2. Endereço do GitHub

base_url = (
    "https://raw.githubusercontent.com/"
    "thalesfreirefarias/Controle-Financeiro-Pro-/main/"
)


# 3. Importação do CSV

df = pd.read_csv(
    base_url + "data/movimentacoes.csv",
    parse_dates=["Data"]
)

print("CSV importado com sucesso!")

display(df)


# 4. Importação do Excel

resposta = requests.get(
    base_url + "output/Controle_Financeiro_Pro.xlsx",
    timeout=30
)

resposta.raise_for_status()

wb = load_workbook(BytesIO(resposta.content))

print("Excel importado com sucesso!")
print("Abas:", wb.sheetnames)

CSV importado com sucesso!


,Data,Descrição,Tipo,Categoria,Pagamento,Valor
0,2026-11-05,Salário,Entrada,Salário,Transferência,5000.0
1,2026-11-06,Supermercado,Saída,Alimentação,Pix,250.0
2,2026-11-08,Restaurante,Saída,Lazer,Crédito,150.0
3,2026-11-10,Aluguel,Saída,Moradia,Pix,1500.0
4,2026-11-15,Freelance,Entrada,Renda Extra,Transferência,800.0
5,2026-11-20,Internet,Saída,Serviços,Débito,120.0


Excel importado com sucesso!
Abas: ['Dashboard', 'Lançamentos', 'Orçamento', 'Metas', 'Configurações']


In [2]:
# 5. Identidade visual

cores = {
    "fundo": "0F172A",
    "painel": "1E293B",
    "receitas": "10B981",
    "despesas": "EF4444",
    "destaque": "38BDF8",
    "texto": "F8FAFC"
}


# 6. Selecionar a aba Lançamentos

ws_lanc = wb["Lançamentos"]

ws_lanc.sheet_view.showGridLines = False


# 7. Inserir os cabeçalhos

ws_lanc.append(df.columns.tolist())


# 8. Inserir as movimentações

for linha in df.itertuples(index=False, name=None):
    ws_lanc.append(linha)


# 9. Formatação dos cabeçalhos

for celula in ws_lanc[1]:

    celula.fill = PatternFill(
        fill_type="solid",
        fgColor=cores["fundo"]
    )

    celula.font = Font(
        bold=True,
        color=cores["texto"]
    )

    celula.alignment = Alignment(
        horizontal="center"
    )


# 10. Largura das colunas

larguras = {
    "A": 16,
    "B": 30,
    "C": 15,
    "D": 22,
    "E": 22,
    "F": 18
}

for coluna, largura in larguras.items():
    ws_lanc.column_dimensions[coluna].width = largura


# 11. Formatação de datas e valores

for linha in range(2, 1001):

    ws_lanc[f"A{linha}"].number_format = "dd/mm/yyyy"

    ws_lanc[f"F{linha}"].number_format = '"R$" #,##0.00'


# 12. Criar tabela estruturada

tabela = Table(
    displayName="TabelaMovimentacoes",
    ref=f"A1:F{len(df) + 1}"
)

tabela.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

ws_lanc.add_table(tabela)


# 13. Listas suspensas

listas = {
    "C": "Entrada,Saída",

    "D": (
        "Salário,Renda Extra,Investimentos,"
        "Alimentação,Moradia,Transporte,"
        "Saúde,Lazer,Educação,Serviços,"
        "Compras,Outros"
    ),

    "E": (
        "Pix,Crédito,Débito,"
        "Dinheiro,Transferência"
    )
}

for coluna, opcoes in listas.items():

    validacao = DataValidation(
        type="list",
        formula1=f'"{opcoes}"',
        allow_blank=True
    )

    validacao.showErrorMessage = True
    validacao.errorTitle = "Valor inválido"
    validacao.error = "Selecione uma opção da lista."

    ws_lanc.add_data_validation(validacao)

    validacao.add(f"{coluna}2:{coluna}1000")


# 14. Validação dos valores financeiros

validacao_valor = DataValidation(
    type="decimal",
    operator="greaterThan",
    formula1=0,
    allow_blank=True
)

validacao_valor.showErrorMessage = True
validacao_valor.error = "Informe um valor maior que zero."

ws_lanc.add_data_validation(validacao_valor)

validacao_valor.add("F2:F1000")


# 15. Congelar o cabeçalho

ws_lanc.freeze_panes = "A2"

print("Aba Lançamentos criada com sucesso!")

Aba Lançamentos criada com sucesso!


In [3]:
from google.colab import files

wb.save("/content/Controle_Financeiro_Teste.xlsx")

files.download("/content/Controle_Financeiro_Teste.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# ==========================================
# CONTROLE FINANCEIRO PRO 2.0
# DIA 3 - BLOCO 4
# DASHBOARD COMPLETO
# ==========================================

from openpyxl.chart import (
    BarChart,
    DoughnutChart,
    LineChart,
    Reference
)

from openpyxl.chart.series import DataPoint
from openpyxl.chart.shapes import GraphicalProperties
from openpyxl.chart.label import DataLabelList
from openpyxl.chart.text import RichText

from openpyxl.drawing.text import (
    Paragraph,
    ParagraphProperties,
    CharacterProperties,
    RegularTextRun
)

from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment
)

from openpyxl.workbook.properties import CalcProperties


# ==========================================
# 22. PREPARAÇÃO DO DASHBOARD
# ==========================================

ws = wb["Dashboard"]
ws_aux = wb["Configurações"]

# Remover gráficos anteriores para evitar duplicação

ws._charts.clear()

# Ampliar a área do dashboard

for linha in ws["A12:Q57"]:
    for celula in linha:
        celula.fill = PatternFill(
            "solid",
            fgColor=cores["fundo"]
        )

for coluna in "ABCDEFGHIJKLMNOPQ":
    ws.column_dimensions[coluna].width = 14

for linha in range(12, 58):
    ws.row_dimensions[linha].height = 20

ws.sheet_view.showGridLines = False
ws.sheet_view.zoomScale = 80


# ==========================================
# 23. IDENTIDADE VISUAL DOS GRÁFICOS
# ==========================================

paleta = [
    "38BDF8",
    "10B981",
    "F59E0B",
    "A78BFA",
    "EF4444",
    "06B6D4",
    "EC4899",
    "84CC16",
    "F97316",
    "6366F1",
    "14B8A6",
    "94A3B8"
]


def texto_grafico(tamanho=1100):

    propriedades = CharacterProperties(
        sz=tamanho,
        solidFill=cores["texto"]
    )

    paragrafo = Paragraph(
        pPr=ParagraphProperties(
            defRPr=propriedades
        ),
        endParaRPr=propriedades
    )

    return RichText(p=[paragrafo])


def estilizar_grafico(grafico):

    grafico.width = 19
    grafico.height = 13

    grafico.graphical_properties = GraphicalProperties(
        solidFill=cores["painel"]
    )

    grafico.graphical_properties.line.noFill = True

    grafico.txPr = texto_grafico()

    if grafico.title:

        for paragrafo in grafico.title.tx.rich.p:

            for trecho in paragrafo.r:

                trecho.rPr = CharacterProperties(
                    sz=1400,
                    b=True,
                    solidFill=cores["texto"]
                )

    if grafico.legend:
        grafico.legend.txPr = texto_grafico(1000)

    for eixo in [
        getattr(grafico, "x_axis", None),
        getattr(grafico, "y_axis", None)
    ]:

        if eixo is not None:
            eixo.txPr = texto_grafico(1000)

            eixo.spPr = GraphicalProperties()

            eixo.spPr.line.solidFill = "475569"

    eixo_y = getattr(grafico, "y_axis", None)

    if eixo_y is not None:

        eixo_y.numFmt = '#,##0'

        if eixo_y.majorGridlines is not None:

            eixo_y.majorGridlines.spPr = (
                GraphicalProperties()
            )

            eixo_y.majorGridlines.spPr.line.solidFill = (
                "334155"
            )


# ==========================================
# 24. BASE AUXILIAR - RECEITAS E DESPESAS
# ==========================================

ws_aux["A1"] = "Tipo"
ws_aux["B1"] = "Valor"

ws_aux["A2"] = "Receitas"
ws_aux["B2"] = "=Dashboard!B8"

ws_aux["A3"] = "Despesas"
ws_aux["B3"] = "=Dashboard!F8"


# ==========================================
# 25. GRÁFICO 1 - RECEITAS X DESPESAS
# ==========================================

grafico1 = BarChart()

grafico1.type = "col"
grafico1.title = "Receitas x Despesas"
grafico1.legend = None
grafico1.gapWidth = 100

grafico1.y_axis.scaling.min = 0

dados = Reference(
    ws_aux,
    min_col=2,
    min_row=1,
    max_row=3
)

rotulos = Reference(
    ws_aux,
    min_col=1,
    min_row=2,
    max_row=3
)

grafico1.add_data(
    dados,
    titles_from_data=True
)

grafico1.set_categories(rotulos)

for indice, cor in enumerate([
    cores["receitas"],
    cores["despesas"]
]):

    ponto = DataPoint(idx=indice)

    ponto.graphicalProperties.solidFill = cor
    ponto.graphicalProperties.line.noFill = True

    grafico1.series[0].data_points.append(ponto)

grafico1.dataLabels = DataLabelList()
grafico1.dataLabels.showVal = True
grafico1.dataLabels.showLegendKey = False
grafico1.dataLabels.txPr = texto_grafico()

estilizar_grafico(grafico1)

ws.add_chart(grafico1, "B13")


# ==========================================
# 26. BASE AUXILIAR - CATEGORIAS
# ==========================================

categorias = [
    "Salário",
    "Renda Extra",
    "Investimentos",
    "Alimentação",
    "Moradia",
    "Transporte",
    "Saúde",
    "Lazer",
    "Educação",
    "Serviços",
    "Compras",
    "Outros"
]

ws_aux["D1"] = "Categoria"
ws_aux["E1"] = "Despesas"

for linha, categoria in enumerate(
    categorias,
    start=2
):

    ws_aux[f"D{linha}"] = categoria

    ws_aux[f"E{linha}"] = (
        "=SUMIFS("
        "'Lançamentos'!$F$2:$F$1000,"
        "'Lançamentos'!$C$2:$C$1000,"
        '"Saída",'
        "'Lançamentos'!$D$2:$D$1000,"
        f"D{linha},"
        "'Lançamentos'!$A$2:$A$1000,"
        '">="&DATE(Dashboard!$G$5,'
        'Dashboard!$C$5,1),'
        "'Lançamentos'!$A$2:$A$1000,"
        '"<"&DATE(Dashboard!$G$5,'
        'Dashboard!$C$5+1,1))'
    )


# ==========================================
# 27. GRÁFICO 2 - DESPESAS POR CATEGORIA
# ==========================================

grafico2 = DoughnutChart()

grafico2.title = "Despesas por Categoria"
grafico2.holeSize = 65
grafico2.firstSliceAng = 270

dados = Reference(
    ws_aux,
    min_col=5,
    min_row=1,
    max_row=13
)

rotulos = Reference(
    ws_aux,
    min_col=4,
    min_row=2,
    max_row=13
)

grafico2.add_data(
    dados,
    titles_from_data=True
)

grafico2.set_categories(rotulos)

for indice, cor in enumerate(paleta):

    ponto = DataPoint(idx=indice)

    ponto.graphicalProperties.solidFill = cor
    ponto.graphicalProperties.line.noFill = True

    grafico2.series[0].data_points.append(ponto)

grafico2.legend.position = "r"

estilizar_grafico(grafico2)

ws.add_chart(grafico2, "J13")


# ==========================================
# 28. BASE AUXILIAR - EVOLUÇÃO ANUAL
# ==========================================

cabecalhos = [
    "Mês",
    "Receitas",
    "Despesas",
    "Saldo",
    "Saldo Acumulado"
]

for coluna, titulo in enumerate(
    cabecalhos,
    start=7
):

    ws_aux.cell(
        row=1,
        column=coluna,
        value=titulo
    )


meses = [
    "Jan", "Fev", "Mar", "Abr",
    "Mai", "Jun", "Jul", "Ago",
    "Set", "Out", "Nov", "Dez"
]


def formula_mensal(tipo, mes):

    return (
        "=SUMIFS("
        "'Lançamentos'!$F$2:$F$1000,"
        "'Lançamentos'!$C$2:$C$1000,"
        f'"{tipo}",'
        "'Lançamentos'!$A$2:$A$1000,"
        f'">="&DATE(Dashboard!$G$5,{mes},1),'
        "'Lançamentos'!$A$2:$A$1000,"
        f'"<"&DATE(Dashboard!$G$5,{mes+1},1))'
    )


for mes, nome in enumerate(meses, start=1):

    linha = mes + 1

    ws_aux[f"G{linha}"] = nome

    ws_aux[f"H{linha}"] = formula_mensal(
        "Entrada",
        mes
    )

    ws_aux[f"I{linha}"] = formula_mensal(
        "Saída",
        mes
    )

    ws_aux[f"J{linha}"] = (
        f"=H{linha}-I{linha}"
    )

    ws_aux[f"K{linha}"] = (
        f"=SUM($J$2:J{linha})"
    )


# ==========================================
# 29. GRÁFICO 3 - EVOLUÇÃO FINANCEIRA
# ==========================================

grafico3 = BarChart()

grafico3.type = "col"
grafico3.grouping = "clustered"
grafico3.overlap = 0

grafico3.title = "Evolução Financeira Anual"

grafico3.y_axis.scaling.min = 0

dados = Reference(
    ws_aux,
    min_col=8,
    max_col=9,
    min_row=1,
    max_row=13
)

rotulos = Reference(
    ws_aux,
    min_col=7,
    min_row=2,
    max_row=13
)

grafico3.add_data(
    dados,
    titles_from_data=True
)

grafico3.set_categories(rotulos)

grafico3.series[0].graphicalProperties.solidFill = (
    cores["receitas"]
)

grafico3.series[1].graphicalProperties.solidFill = (
    cores["despesas"]
)

for serie in grafico3.series:

    serie.graphicalProperties.line.noFill = True

grafico3.legend.position = "b"

estilizar_grafico(grafico3)

ws.add_chart(grafico3, "B34")


# ==========================================
# 30. GRÁFICO 4 - SALDO ACUMULADO
# ==========================================

grafico4 = LineChart()

grafico4.title = "Saldo Acumulado no Ano"
grafico4.legend = None

dados = Reference(
    ws_aux,
    min_col=11,
    min_row=1,
    max_row=13
)

rotulos = Reference(
    ws_aux,
    min_col=7,
    min_row=2,
    max_row=13
)

grafico4.add_data(
    dados,
    titles_from_data=True
)

grafico4.set_categories(rotulos)

serie = grafico4.series[0]

serie.graphicalProperties.line.solidFill = (
    cores["destaque"]
)

serie.graphicalProperties.line.width = 30000

serie.marker.symbol = "circle"
serie.marker.size = 5

serie.marker.graphicalProperties.solidFill = (
    cores["destaque"]
)

serie.marker.graphicalProperties.line.solidFill = (
    cores["destaque"]
)

serie.smooth = False

estilizar_grafico(grafico4)

ws.add_chart(grafico4, "J34")


# ==========================================
# 31. FORMATAÇÃO DA BASE AUXILIAR
# ==========================================

for celula in ws_aux[1]:

    celula.fill = PatternFill(
        "solid",
        fgColor=cores["fundo"]
    )

    celula.font = Font(
        bold=True,
        color=cores["texto"]
    )

for coluna in "ABCDEFGHIJK":

    ws_aux.column_dimensions[coluna].width = 22

for linha in range(2, 14):

    for coluna in ["B", "E", "H", "I", "J", "K"]:

        ws_aux[f"{coluna}{linha}"].number_format = (
            '"R$" #,##0.00'
        )

ws_aux.freeze_panes = "A2"


# ==========================================
# 32. RODAPÉ DO DASHBOARD
# ==========================================

ws.merge_cells("B55:P56")

ws["B55"] = (
    "CONTROLE FINANCEIRO PRO 2.0 | "
    "PERSONAL FINANCE ANALYTICS"
)

ws["B55"].font = Font(
    size=10,
    color="94A3B8"
)

ws["B55"].alignment = Alignment(
    horizontal="center",
    vertical="center"
)


# ==========================================
# 33. CONFIGURAÇÕES FINAIS
# ==========================================

wb.calculation = CalcProperties(
    calcMode="auto",
    fullCalcOnLoad=True,
    forceFullCalc=True
)

wb.active = wb.sheetnames.index("Dashboard")

print("Dashboard completo criado com sucesso!")
print("Quantidade de gráficos:", len(ws._charts))
print("Indicadores dinâmicos: 4")
print("Gráficos dinâmicos: 4")

Dashboard completo criado com sucesso!
Quantidade de gráficos: 4
Indicadores dinâmicos: 4
Gráficos dinâmicos: 4


In [7]:
from google.colab import files

wb.save("/content/Controle_Financeiro_Teste.xlsx")

files.download("/content/Controle_Financeiro_Teste.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# ==========================================
# DIA 3 - PUBLICAÇÃO DA VERSÃO V0.1.0
# ==========================================

import requests
import base64

from io import BytesIO
from google.colab import userdata

# Configuração do GitHub

token = userdata.get("GITHUB_TOKEN")

repositorio = "thalesfreirefarias/Controle-Financeiro-Pro-"

caminho = "output/Controle_Financeiro_Pro_V1.xlsx"

url = (
    f"https://api.github.com/repos/"
    f"{repositorio}/contents/{caminho}"
)

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}

# Salvar o Excel em memória

arquivo = BytesIO()

wb.save(arquivo)

conteudo = base64.b64encode(
    arquivo.getvalue()
).decode("utf-8")

# Verificar se o arquivo já existe

consulta = requests.get(
    url,
    headers=headers,
    params={"ref": "main"},
    timeout=30
)

payload = {
    "message": "feat: release financial dashboard v0.1.0",
    "content": conteudo,
    "branch": "main"
}

if consulta.status_code == 200:
    payload["sha"] = consulta.json()["sha"]

elif consulta.status_code != 404:
    consulta.raise_for_status()

# Publicar o Excel

resposta = requests.put(
    url,
    headers=headers,
    json=payload,
    timeout=30
)

resposta.raise_for_status()

print("Versão v0.1.0 publicada com sucesso!")
print(resposta.json()["content"]["html_url"])

Versão v0.1.0 publicada com sucesso!
https://github.com/thalesfreirefarias/Controle-Financeiro-Pro-/blob/main/output/Controle_Financeiro_Pro_V1.xlsx
